# Лабораторная работа 3. О линейной алгебре в Звёздных Войнах.

## Введение

Инженеры Альянса повстанцев разрабатывают навигационный модуль для управления флотом.
Положение кораблей, переходы между системами координат, калибровка датчиков и анализ устойчивости автопилота описываются с помощью методов линейной алгебры.

В этой лабораторной работе необходимо использовать:

- `numpy` и `numpy.linalg` для численных вычислений;
- `sympy` для символьных вычислений и анализа параметризованных систем.

# Задача 1. Градиентный спуск по космосу. (2 балла)

Наш модуль умеет оценивать вероятность нахождения вражеского флота для каждой точки. Напиши простой градиентный спуск при помощи `sympy`, чтобы найти точку, где вероятность нахождения вражеский флота **наименьшая**, чтобы мы могли отправить туда корабли с гражданскими.

### P.S. напоминание определения.

Градиентный спуск - итеративный алгоритм, который позволяет искать локальный минимум/максимум функции.

Идея: а давайте каждый раз делать микро-шаги в сторону антиградиента:

`steps` раз обнови текущую точку по формуле: $x_{t+1} = x_t - \gamma \nabla F(x_t)$, где $\gamma$ - learning rate (lr). 

$x_0$ - стартовая точка, задаётся через параметр `start`. `expr` - выражение, для которого запускается градиентный спуск; `variables` - переменные, по которым идёт пересчёт (их может быть несколько).

In [1]:
import sympy as sp


def gradient_descent(expr, variables, start, lr=0.1, steps=100):
    grads = [sp.diff(expr, var) for var in variables]
    point = [float(x) for x in start]

    for _ in range(steps):
        subs = {var: point[i] for i, var in enumerate(variables)}
        grad_vals = [float(g.evalf(subs=subs)) for g in grads]
        point = [point[i] - lr * grad_vals[i] for i in range(len(point))]

    return point


In [ ]:
# =========================
# Испытания модуля (тесты)
# =========================

def test_quadratic_1d():
    x = sp.Symbol('x')
    expr = (x - 3)**2
    result = gradient_descent(expr, [x], start=[0], lr=0.1, steps=200)
    # Минимум около на х = 3
    assert abs(result[0] - 3) < 1e-3


def test_quadratic_2d():
    # 2д карта
    x, y = sp.symbols('x y')
    expr = (x - 1)**2 + (y + 2)**2  # Минимум на (1, -2)
    result = gradient_descent(expr, [x, y], start=[0, 0], lr=0.1, steps=200)
    assert abs(result[0] - 1) < 1e-3
    assert abs(result[1] + 2) < 1e-3


def test_flat_minimum():
    x = sp.Symbol('x')
    expr = x**4
    result = gradient_descent(expr, [x], start=[2], lr=0.05, steps=50000)
    assert abs(result[0]) < 1e-2


def test_already_optimal():
    x = sp.Symbol('x')
    expr = (x - 5)**2
    result = gradient_descent(expr, [x], start=[5], lr=0.1, steps=10)
    assert abs(result[0] - 5) < 1e-9

test_quadratic_1d()
test_quadratic_2d()
test_flat_minimum()
test_already_optimal()

## Задача 2. Калибровка навигационного модуля X-wing (2 балла)

Инженеры Альянса повстанцев обнаружили, что навигационный модуль истребителя **X-wing** выполняет линейное преобразование координат сенсоров.  
Это преобразование описывается матрицей

$$
A =
\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
$$

Чтобы восстановить исходные координаты объектов в пространстве, необходимо вычислить **обратное преобразование**, то есть **обратную матрицу** $A^{-1}$.

Обратная матрица должна удовлетворять условию

$$
A \cdot A^{-1} = I,
$$

где $I$ — единичная матрица.

Для матрицы

$$
\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}
$$

обратная существует **только если её определитель не равен нулю**:

$$
\det(A) = ad - bc \ne 0.
$$

В этом случае обратная матрица вычисляется по формуле

$$
A^{-1} =
\frac{1}{ad-bc}
\begin{pmatrix}
d & -b \\
-c & a
\end{pmatrix}.
$$

### Задание

Напишите функцию на Python, которая:

1. принимает **матрицу $2 \times 2$** в виде списка списков;
2. вычисляет её **обратную матрицу**;
3. возвращает `None`, если матрица **необратима** (то есть её определитель равен нулю).

In [ ]:
def inverse_2x2(matrix: list[list[float]]) -> list[list[float]] | None:
    """
    Calculate the inverse of a 2x2 matrix.

    Args:
        matrix: A 2x2 matrix represented as [[a, b], [c, d]]

    Returns:
        The inverse matrix as a 2x2 list, or None if the matrix is singular
        (i.e., determinant equals zero)
    """
    a, b = matrix[0]
    c, d = matrix[1]

    det = a * d - b * c
    if det == 0:
        return None

    return [[d / det, -b / det],
            [-c / det, a / det]]


## Задача 3. Анализ сигналов навигационного сенсора (2 балла)

Навигационные системы кораблей Альянса получают сигналы от сенсоров, которые можно представить в виде **векторов** или **матриц измерений**.  
Чтобы оценить силу сигнала или величину ошибки измерений, инженеры используют **нормы**.

Норма — это численная характеристика, которая измеряет «размер» или «длину» вектора (или матрицы).

В этой задаче нужно реализовать вычисление нескольких типов норм, часто используемых в линейной алгебре и машинном обучении.

### Типы норм

Пусть задан массив элементов $x_1, x_2, \dots, x_n$.

**L1-норма**

$$
\|x\|_1 = \sum_{i=1}^{n} |x_i|
$$

**L2-норма**

$$
\|x\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}
$$

**Норма Фробениуса**

Для матрицы $A$:

$$
\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2}
$$


### Задание

Реализуйте функцию

```python
compute_norm(arr, norm_type)

In [ ]:
import numpy as np

def compute_norm(arr: np.ndarray, norm_type: str) -> float:
    """
    Compute the specified norm of the input array.

    Args:
        arr: Input numpy array (1D or 2D)
        norm_type: Type of norm ('l1', 'l2', or 'frobenius')

    Returns:
        The computed norm as a float
    """
    arr = np.array(arr, dtype=float)

    if norm_type == 'l1':
        return float(np.sum(np.abs(arr)))
    else:
        return float(np.sqrt(np.sum(arr ** 2)))

## Задача 4. Декомпозиция матрицы навигационной системы (LU-разложение) (3 балла).

Навигационный компьютер **звёздного разрушителя Империи** использует матричную модель для расчёта траектории флота.  
Матрица коэффициентов системы обозначается

$$
A.
$$

Для ускорения вычислений эта матрица раскладывается в произведение двух более простых матриц:

$$
A = LU,
$$

где

- $L$ — **нижнетреугольная матрица** (элементы выше диагонали равны нулю),
- $U$ — **верхнетреугольная матрица** (элементы ниже диагонали равны нулю).

Такое разложение называется **LU-разложением** и широко используется при решении систем линейных уравнений.

В этой задаче необходимо реализовать LU-разложение по **алгоритму Дулиттла**.

### Алгоритм Дулиттла

В алгоритме Дулиттла:

- диагональные элементы матрицы $L$ равны 1:

$$
L_{ii} = 1
$$

- элементы матрицы $U$ вычисляются по формуле

$$
U_{ij} =
A_{ij} -
\sum_{k=1}^{i-1} L_{ik}U_{kj}
$$

- элементы матрицы $L$ вычисляются как

$$
L_{ij} =
\frac{1}{U_{jj}}
\left(
A_{ij} -
\sum_{k=1}^{j-1} L_{ik}U_{kj}
\right)
\quad \text{для } i > j
$$

### Задание

Реализуйте функцию

```python
lu_decomposition(A)

In [5]:
import numpy as np

def lu_decomposition(A: list) -> tuple:
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i, n):
            U[i][j] = A[i][j] - sum(L[i][k] * U[k][j] for k in range(i))

        for j in range(i + 1, n):

            L[j][i] = (A[j][i] - sum(L[j][k] * U[k][i] for k in range(i))) / U[i][i]

    return L, U

In [6]:
# TODO - напиши тесты, сравни с numpy.linalg.lu

## Задача 5. Проверка стабильности навигационного фильтра (разложение Холецкого) (3 балла).

Навигационные системы кораблей **Галактической Империи** используют статистические модели для обработки данных сенсоров.  
Ковариационная матрица ошибок измерений обозначается

$$
A.
$$

Для ускорения вычислений и проверки корректности модели используется **разложение Холецкого**.

### Разложение Холецкого

Разложение Холецкого применяется к **симметричным положительно определённым матрицам**.

Если матрица $A$ обладает этими свойствами, её можно представить в виде

$$
A = L \cdot L^{T},
$$

где

- $L$ — **нижнетреугольная матрица**,
- $L^{T}$ — **транспонированная матрица**.


### Формулы вычисления

Элементы матрицы $L$ вычисляются последовательно.

Для диагональных элементов:

$$
L_{ii} =
\sqrt{
A_{ii} -
\sum_{k=1}^{i-1} L_{ik}^{2}
}
$$

Для элементов ниже диагонали:

$$
L_{ij} =
\frac{
A_{ij} -
\sum_{k=1}^{j-1} L_{ik}L_{jk}
}{
L_{jj}
}
\quad \text{при } i > j
$$

Элементы выше диагонали равны нулю.


### Задание

Реализуйте функцию

```python
cholesky_decomposition(A)
````

которая выполняет **разложение Холецкого** матрицы $A$.

Функция должна:

* принимать на вход **симметричную положительно определённую матрицу**
  (двумерный список или массив `numpy`);
* вычислять **нижнетреугольную матрицу** $L$;
* возвращать $L$ как **двумерный список чисел типа float**.


Использование функций вида

```
numpy.linalg.cholesky
```

**запрещено**.


### Некорректные входные данные

Функция должна вернуть

```
-1
```

если:

* матрица **не квадратная**;
* матрица **пустая** или имеет неверную структуру;
* матрица **не является положительно определённой**.

Если во время вычислений под знаком корня возникает **отрицательное число**, это означает, что матрица не является положительно определённой, и разложение невозможно.


### Интерпретация

Разложение Холецкого используется в навигационных системах для эффективной работы с ковариационными матрицами ошибок сенсоров.
Этот метод широко применяется в:

* решении систем линейных уравнений,
* фильтрах Калмана,
* статистическом моделировании,
* вычислении обратных матриц.



In [ ]:
import numpy as np

def cholesky_decomposition(A):
    """
    Perform Cholesky decomposition on a symmetric positive-definite matrix.

    Args:
        A: A symmetric positive-definite matrix (2D list or numpy array)

    Returns:
        L: Lower triangular matrix such that A = L @ L.T as a 2D list,
           or -1 if decomposition is not possible
    """
    try:
        A = np.array(A, dtype=float)
    except:
        return -1

    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        return -1

    n = A.shape[0]

    if not np.allclose(A, A.T):
        return -1

    L = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1):
            if i == j:
                value = A[i, i] - sum(L[i, k] ** 2 for k in range(j))
                if value <= 0:
                    return -1
                L[i, j] = np.sqrt(value)
            else:
                L[i, j] = (A[i, j] - sum(L[i, k] * L[j, k] for k in range(j))) / L[j, j]

    return L.tolist()

In [8]:
def test_cholesky_2x2():
    A = [[4, 2],
         [2, 3]]

    L = np.array(cholesky_decomposition(A), dtype=float)
    A = np.array(A, dtype=float)
    L_np = np.linalg.cholesky(A)

    assert np.allclose(A, L @ L.T)
    assert np.allclose(L, np.tril(L))
    assert np.allclose(L, L_np)


def test_cholesky_3x3():
    A = [[25, 15, -5],
         [15, 18,  0],
         [-5,  0, 11]]

    L = np.array(cholesky_decomposition(A), dtype=float)
    A = np.array(A, dtype=float)
    L_np = np.linalg.cholesky(A)

    assert np.allclose(A, L @ L.T)
    assert np.allclose(L, np.tril(L))
    assert np.allclose(L, L_np)


def test_cholesky_identity():
    A = [[1, 0, 0],
         [0, 1, 0],
         [0, 0, 1]]

    L = np.array(cholesky_decomposition(A), dtype=float)
    A = np.array(A, dtype=float)
    L_np = np.linalg.cholesky(A)

    assert np.allclose(A, L @ L.T)
    assert np.allclose(L, np.eye(3))
    assert np.allclose(L, L_np)


def test_cholesky_not_square():
    A = [[1, 2, 3],
         [4, 5, 6]]

    assert cholesky_decomposition(A) == -1


def test_cholesky_not_symmetric():
    A = [[1, 2],
         [3, 4]]

    assert cholesky_decomposition(A) == -1


def test_cholesky_not_positive_definite():
    A = [[1, 2],
         [2, 1]]

    assert cholesky_decomposition(A) == -1


def test_cholesky_empty():
    A = []

    assert cholesky_decomposition(A) == -1


test_cholesky_2x2()
test_cholesky_3x3()
test_cholesky_identity()
test_cholesky_not_square()
test_cholesky_not_symmetric()
test_cholesky_not_positive_definite()
test_cholesky_empty()

## Задача 6. Анализ сигнала дальнего сканера (SVD-разложение) (3 балла).

Дальний сканер корабля **Millennium Falcon** регистрирует двумерный сигнал от удалённых объектов.  
Полученные данные можно представить в виде матрицы

$$
A =
\begin{pmatrix}
a_{11} & a_{12} \\
a_{21} & a_{22}
\end{pmatrix}.
$$

Для анализа структуры сигнала инженеры используют **сингулярное разложение (SVD)**.  
Оно позволяет выделить основные направления сигнала и его интенсивность.


### Сингулярное разложение

Любая вещественная матрица может быть представлена в виде

$$
A = U \, \Sigma \, V^{T},
$$

где

- $U$ — **ортогональная матрица** левых сингулярных векторов,
- $V$ — **ортогональная матрица** правых сингулярных векторов,
- $\Sigma$ — диагональная матрица сингулярных значений

$$
\Sigma =
\begin{pmatrix}
\sigma_1 & 0 \\
0 & \sigma_2
\end{pmatrix}.
$$

### Метод Якоби

Для приближённого вычисления SVD можно использовать **вращение Якоби**.

Матрица вращения имеет вид

$$
J =
\begin{pmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{pmatrix}.
$$

Одно вращение позволяет приблизительно диагонализовать матрицу $A^TA$, что даёт оценку сингулярных значений.

В этой задаче требуется выполнить **один шаг вращения Якоби** (без последующих итераций).

### Задание

Реализуйте функцию

```python
svd_2x2(A)
````

которая:

* принимает матрицу `A` размера **2×2** (`numpy.ndarray`);
* вычисляет **приближённое сингулярное разложение**;
* выполняет **один шаг вращения Якоби**.

Использовать можно только **базовые операции NumPy**:

* умножение матриц,
* транспонирование,
* элемент-wise операции.

Использование функций вида

```
numpy.linalg.svd
```

**запрещено**.

### Возвращаемое значение

Функция должна вернуть кортеж

```python
(U, S, Vt)
```

где

* `U` — ортогональная матрица $2 \times 2$ (левые сингулярные векторы),
* `S` — массив длины 2 со **сингулярными значениями**,
* `Vt` — транспонированная матрица правых сингулярных векторов.

Приближённое равенство должно выполняться:

$$
A \approx U \cdot \mathrm{diag}(S) \cdot V^{T}.
$$


### Интерпретация

SVD используется для анализа структуры сигналов и выделения главных направлений данных.
В навигационных системах космических кораблей этот метод помогает:

* отделять полезный сигнал от шума,
* выполнять сжатие данных сенсоров,
* анализировать геометрию наблюдаемых объектов.



In [9]:
import numpy as np

def svd_2x2_singular_values(A: np.ndarray) -> tuple:
    A = np.array(A, dtype=float)

    B = A.T @ A
    a, b = B[0, 0], B[0, 1]
    d = B[1, 1]

    if abs(b) < 1e-12:
        c, s = 1.0, 0.0
    else:
        theta = 0.5 * np.arctan2(2 * b, a - d)
        c, s = np.cos(theta), np.sin(theta)

    V = np.array([[c, -s],
                  [s,  c]])

    D = V.T @ B @ V
    eigvals = np.diag(D)
    eigvals = np.maximum(eigvals, 0.0)
    S = np.sqrt(eigvals)

    order = np.argsort(-S)
    S = S[order]
    V = V[:, order]

    U = np.zeros((2, 2))
    for i in range(2):
        if S[i] > 1e-12:
            U[:, i] = (A @ V[:, i]) / S[i]
        else:
            U[:, i] = np.array([1.0, 0.0]) if i == 0 else np.array([0.0, 1.0])

    return U, S, V.T

In [10]:
import numpy as np

def test_svd_2x2_diagonal():
    A = np.array([[3.0, 0.0],
                  [0.0, 2.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    S = np.array(S, dtype=float)
    Vt = np.array(Vt, dtype=float)

    U_np, S_np, Vt_np = np.linalg.svd(A)

    assert np.allclose(sorted(S, reverse=True), S_np, atol=1e-6)
    assert np.allclose(A, U @ np.diag(S) @ Vt, atol=1e-6)


def test_svd_2x2_symmetric():
    A = np.array([[4.0, 1.0],
                  [1.0, 3.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    S = np.array(S, dtype=float)
    Vt = np.array(Vt, dtype=float)

    U_np, S_np, Vt_np = np.linalg.svd(A)

    assert np.allclose(sorted(S, reverse=True), S_np, atol=1e-5)
    assert np.allclose(A, U @ np.diag(S) @ Vt, atol=1e-5)


def test_svd_2x2_general():
    A = np.array([[1.0, 2.0],
                  [3.0, 4.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    S = np.array(S, dtype=float)
    Vt = np.array(Vt, dtype=float)

    U_np, S_np, Vt_np = np.linalg.svd(A)

    assert np.allclose(sorted(S, reverse=True), S_np, atol=1e-4)
    assert np.allclose(A, U @ np.diag(S) @ Vt, atol=1e-4)


def test_svd_rank_1():
    A = np.array([[2.0, 4.0],
                  [1.0, 2.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    S = np.array(S, dtype=float)
    Vt = np.array(Vt, dtype=float)

    U_np, S_np, Vt_np = np.linalg.svd(A)

    assert np.allclose(sorted(S, reverse=True), S_np, atol=1e-5)
    assert np.allclose(A, U @ np.diag(S) @ Vt, atol=1e-5)


def test_svd_zero_matrix():
    A = np.array([[0.0, 0.0],
                  [0.0, 0.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    S = np.array(S, dtype=float)
    Vt = np.array(Vt, dtype=float)

    U_np, S_np, Vt_np = np.linalg.svd(A)

    assert np.allclose(sorted(S, reverse=True), S_np, atol=1e-8)
    assert np.allclose(A, U @ np.diag(S) @ Vt, atol=1e-8)


def test_svd_orthogonality():
    A = np.array([[5.0, 2.0],
                  [2.0, 1.0]])

    U, S, Vt = svd_2x2_singular_values(A)
    U = np.array(U, dtype=float)
    Vt = np.array(Vt, dtype=float)

    assert np.allclose(U.T @ U, np.eye(2), atol=1e-5)
    assert np.allclose(Vt @ Vt.T, np.eye(2), atol=1e-5)


test_svd_2x2_diagonal()
test_svd_2x2_symmetric()
test_svd_2x2_general()
test_svd_rank_1()
test_svd_zero_matrix()
test_svd_orthogonality()

## Задача 7. Перекалибровка сенсорной матрицы (QR-разложение) (3 балла)

Перед гиперпространственным прыжком инженеры **Альянса повстанцев** анализируют данные сенсорной матрицы корабля **X-wing**.  
Измерения записываются в виде матрицы

$$
A.
$$

Чтобы выделить независимые направления измерений и упростить дальнейшие вычисления, матрицу представляют в виде **QR-разложения**.

### QR-разложение

Любая матрица $A$ может быть представлена в виде

$$
A = Q R,
$$

где

- $Q$ — **ортогональная матрица**, столбцы которой образуют **ортонормированный базис**  
  $$
  Q^T Q = I
  $$
- $R$ — **верхнетреугольная матрица**.

### Метод Грама–Шмидта

Чтобы построить матрицу $Q$, используется **процесс Грама–Шмидта**.  
Пусть столбцы матрицы $A$ обозначены

$$
a_1, a_2, \dots, a_n.
$$

Тогда ортогональные векторы вычисляются следующим образом.

Первый столбец:

$$
q_1 = \frac{a_1}{\|a_1\|}
$$

Для каждого следующего столбца:

$$
u_k =
a_k -
\sum_{j=1}^{k-1}
\langle a_k, q_j \rangle q_j
$$

$$
q_k = \frac{u_k}{\|u_k\|}
$$

После построения матрицы $Q$ элементы матрицы $R$ определяются как

$$
R = Q^T A.
$$

### Задание

Реализуйте функцию

```python
qr_decomposition(A)
````

которая:

* принимает матрицу `A`;
* выполняет **QR-разложение с использованием процесса Грама–Шмидта**;
* возвращает кортеж

```python
(Q, R)
```

где

* `Q` — матрица со **столбцами-ортонормированными векторами**;
* `R` — **верхнетреугольная матрица**;
* выполняется равенство

$$
A = Q @ R.
$$

### Ограничения

* реализация должна использовать **алгоритм Грама–Шмидта**;
* запрещается использовать готовые функции типа

```
numpy.linalg.qr
```

### Интерпретация

QR-разложение применяется для:

* решения систем линейных уравнений,
* задач наименьших квадратов,
* ортогонализации признаков,
* устойчивых численных алгоритмов.

В навигационных системах космических кораблей QR-разложение помогает отделять независимые направления сигналов сенсоров и стабилизировать вычисления траектории.


In [11]:
import numpy as np

def qr_decomposition(A):
    """
	Perform QR decomposition using Gram-Schmidt process.

	Args:
		A: An m x n matrix

	Returns:
		Tuple of (Q, R) where Q is orthogonal and R is upper triangular
	"""
    A = np.array(A, dtype=float)
    m, n = A.shape

    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)

    for j in range(n):
        v = A[:, j].copy()

        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]

        R[j, j] = np.linalg.norm(v)
        if R[j, j] < 1e-12:
            raise ValueError("Columns are linearly dependent")

        Q[:, j] = v / R[j, j]

    return Q, R

In [12]:
def test_qr_square_matrix():
    A = np.array([[1.0, 2.0],
                  [3.0, 4.0]])

    Q, R = qr_decomposition(A)
    Q = np.array(Q, dtype=float)
    R = np.array(R, dtype=float)

    Q_np, R_np = np.linalg.qr(A)

    assert np.allclose(A, Q @ R, atol=1e-6)
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-6)
    assert np.allclose(R, np.triu(R), atol=1e-6)
    assert np.allclose(A, Q_np @ R_np, atol=1e-6)


def test_qr_3x3_matrix():
    A = np.array([[1.0, 1.0, 0.0],
                  [1.0, 0.0, 1.0],
                  [0.0, 1.0, 1.0]])

    Q, R = qr_decomposition(A)
    Q = np.array(Q, dtype=float)
    R = np.array(R, dtype=float)

    Q_np, R_np = np.linalg.qr(A)

    assert np.allclose(A, Q @ R, atol=1e-6)
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-6)
    assert np.allclose(R, np.triu(R), atol=1e-6)
    assert np.allclose(A, Q_np @ R_np, atol=1e-6)


def test_qr_rectangular_tall():
    A = np.array([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])

    Q, R = qr_decomposition(A)
    Q = np.array(Q, dtype=float)
    R = np.array(R, dtype=float)

    Q_np, R_np = np.linalg.qr(A)

    assert np.allclose(A, Q @ R, atol=1e-6)
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-6)
    assert np.allclose(R, np.triu(R), atol=1e-6)
    assert np.allclose(A, Q_np @ R_np, atol=1e-6)


def test_qr_identity():
    A = np.eye(3)

    Q, R = qr_decomposition(A)
    Q = np.array(Q, dtype=float)
    R = np.array(R, dtype=float)

    Q_np, R_np = np.linalg.qr(A)

    assert np.allclose(A, Q @ R, atol=1e-6)
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-6)
    assert np.allclose(R, np.triu(R), atol=1e-6)
    assert np.allclose(A, Q_np @ R_np, atol=1e-6)


def test_qr_upper_triangular():
    A = np.array([[2.0, 3.0, 1.0],
                  [0.0, 4.0, 5.0],
                  [0.0, 0.0, 6.0]])

    Q, R = qr_decomposition(A)
    Q = np.array(Q, dtype=float)
    R = np.array(R, dtype=float)

    assert np.allclose(A, Q @ R, atol=1e-6)
    assert np.allclose(Q.T @ Q, np.eye(Q.shape[1]), atol=1e-6)
    assert np.allclose(R, np.triu(R), atol=1e-6)


test_qr_square_matrix()
test_qr_3x3_matrix()
test_qr_rectangular_tall()
test_qr_identity()
test_qr_upper_triangular()

# Задача 8. Решение систем линейных уравнений (5 баллов).

Все упомянутые выше разложения (QR, SVD, Cholesky, LU) можно применять для решения систем линейных уравнений.

Напиши функцию 

```python 
solve_linear_system(A: np.array, b: np.array, method)
```

Которая бы находила решение системы Ax=b при помощи заданного разложения.

In [13]:
import numpy as np

def forward_substitution(L, b):
    L = np.array(L, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    y = np.zeros(n, dtype=float)

    for i in range(n):
        if abs(L[i, i]) < 1e-12:
            raise ValueError("Zero diagonal in forward substitution")
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]

    return y


def backward_substitution(U, y):
    U = np.array(U, dtype=float)
    y = np.array(y, dtype=float)
    n = len(y)
    x = np.zeros(n, dtype=float)

    for i in range(n - 1, -1, -1):
        if abs(U[i, i]) < 1e-12:
            raise ValueError("Zero diagonal in backward substitution")
        x[i] = (y[i] - np.dot(U[i, i + 1:], x[i + 1:])) / U[i, i]

    return x


def solve_linear_system(A: np.array, b: np.array, method="qr"):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix")
    if b.ndim != 1 or len(b) != A.shape[0]:
        raise ValueError("b must be a vector of compatible size")

    if method == "lu":
        L, U = lu_decomposition(A.tolist())
        y = forward_substitution(L, b)
        return backward_substitution(U, y)

    if method == "cholesky":
        L = cholesky_decomposition(A)
        if L == -1:
            raise ValueError("Cholesky decomposition is not possible")
        L = np.array(L, dtype=float)
        y = forward_substitution(L, b)
        return backward_substitution(L.T, y)

    if method == "qr":
        Q, R = qr_decomposition(A)
        Q = np.array(Q, dtype=float)
        R = np.array(R, dtype=float)
        y = Q.T @ b
        return backward_substitution(R, y)

    if method == "svd":
        U, S, Vt = svd_2x2_singular_values(A)
        U = np.array(U, dtype=float)
        S = np.array(S, dtype=float)
        Vt = np.array(Vt, dtype=float)

        S_inv = np.zeros_like(S)
        for i in range(len(S)):
            if abs(S[i]) > 1e-12:
                S_inv[i] = 1.0 / S[i]

        return Vt.T @ np.diag(S_inv) @ U.T @ b

    raise ValueError("Unknown method")

In [14]:
def test_solve_lu():
    A = np.array([[4.0, 3.0],
                  [6.0, 3.0]])
    b = np.array([10.0, 12.0])

    x = solve_linear_system(A, b, method="lu")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-6)
    assert np.allclose(A @ x, b, atol=1e-6)


def test_solve_qr():
    A = np.array([[2.0, 1.0],
                  [1.0, 3.0]])
    b = np.array([5.0, 6.0])

    x = solve_linear_system(A, b, method="qr")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-6)
    assert np.allclose(A @ x, b, atol=1e-6)


def test_solve_cholesky():
    A = np.array([[4.0, 2.0],
                  [2.0, 3.0]])
    b = np.array([6.0, 7.0])

    x = solve_linear_system(A, b, method="cholesky")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-6)
    assert np.allclose(A @ x, b, atol=1e-6)


def test_solve_svd():
    A = np.array([[3.0, 1.0],
                  [1.0, 2.0]])
    b = np.array([9.0, 8.0])

    x = solve_linear_system(A, b, method="svd")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-5)
    assert np.allclose(A @ x, b, atol=1e-5)


def test_solve_3x3_lu():
    A = np.array([[2.0, 1.0, 1.0],
                  [4.0, -6.0, 0.0],
                  [-2.0, 7.0, 2.0]])
    b = np.array([5.0, -2.0, 9.0])

    x = solve_linear_system(A, b, method="lu")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-6)
    assert np.allclose(A @ x, b, atol=1e-6)


def test_solve_3x3_qr():
    A = np.array([[1.0, 1.0, 0.0],
                  [1.0, 0.0, 1.0],
                  [0.0, 1.0, 1.0]])
    b = np.array([2.0, 2.0, 2.0])

    x = solve_linear_system(A, b, method="qr")
    x_np = np.linalg.solve(A, b)

    assert np.allclose(x, x_np, atol=1e-6)
    assert np.allclose(A @ x, b, atol=1e-6)


def test_cholesky_invalid():
    A = np.array([[1.0, 2.0],
                  [2.0, 1.0]])
    b = np.array([1.0, 1.0])

    try:
        solve_linear_system(A, b, method="cholesky")
        assert False, "Expected ValueError for non-SPD matrix"
    except ValueError:
        pass


def test_unknown_method():
    A = np.array([[1.0, 0.0],
                  [0.0, 1.0]])
    b = np.array([1.0, 2.0])

    try:
        solve_linear_system(A, b, method="abc")
        assert False, "Expected ValueError for unknown method"
    except ValueError:
        pass


test_solve_lu()
test_solve_qr()
test_solve_cholesky()
test_solve_svd()
test_solve_3x3_lu()
test_solve_3x3_qr()
test_cholesky_invalid()
test_unknown_method()

Теперь проведи детальный анализ вариантов решения. Все ли алгоритмы одинаково точны? Какой алгоритм лучше по времени?

Сравни с `numpy.linalg.solve`.

In [15]:
# TODO

<b><font color="#FF69B4"> Ваш ответ здесь </font></b>

# Бонусные задания.

## Задача Б1. Расчёт параметров навигационного компьютера (метод Гаусса–Зейделя) (3 балла)

Навигационный компьютер корабля **Millennium Falcon** решает систему линейных уравнений, которая возникает при калибровке сенсоров гипердвигателя.  
Эта система имеет вид

$$
A x = b,
$$

где

- $A$ — матрица коэффициентов системы,
- $x$ — вектор неизвестных параметров,
- $b$ — вектор измерений сенсоров.

Для вычисления решения используется **итерационный метод Гаусса–Зейделя**.

### Метод Гаусса–Зейделя

На каждой итерации компоненты решения обновляются последовательно:

$$
x_i^{(k+1)} =
\frac{1}{a_{ii}}
\left(
b_i
-
\sum_{j<i} a_{ij} x_j^{(k+1)}
-
\sum_{j>i} a_{ij} x_j^{(k)}
\right).
$$

Особенность метода заключается в том, что при вычислении очередной компоненты используются **самые свежие значения**, уже обновлённые на текущей итерации.

### Задание

Реализуйте функцию

```python
gauss_seidel(A, b, n, x_ini=None)
````

где

* `A` — квадратная матрица коэффициентов системы,
* `b` — вектор правой части,
* `n` — число итераций метода,
* `x_ini` — начальное приближение для решения (если не задано, используйте нулевой вектор).

Функция должна вернуть **приближённое решение** $x$ после выполнения `n` итераций метода.


### Предположения

Можно считать, что:

* матрица $A$ **диагонально доминирующая**, что обеспечивает сходимость метода;
* все диагональные элементы $a_{ii} \neq 0$;
* система имеет **единственное решение**.


### Интерпретация

Метод Гаусса–Зейделя постепенно уточняет значения вектора $x$, пока приближение не станет достаточно близким к точному решению системы линейных уравнений.

import numpy as np

def gauss_seidel(A, b, n, x_ini=None):
	return np.zeros_like(b)


In [ ]:
import numpy as np

def gauss_seidel(A, b, n, x_ini=None):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x = np.zeros_like(b, dtype=float) if x_ini is None else np.array(x_ini, dtype=float)

    m = len(b)
    for _ in range(n):
        for i in range(m):
            sum1 = np.dot(A[i, :i], x[:i])
            sum2 = np.dot(A[i, i + 1:], x[i + 1:])
            x[i] = (b[i] - sum1 - sum2) / A[i, i]

    return x

## Задача Б2. Навигационный компьютер повстанцев (метод Якоби) (3 балла)

Во время подготовки гиперпространственного прыжка навигационный компьютер базы **Явин IV** должен решить систему линейных уравнений

$$
A x = b,
$$

где

- $A$ — матрица коэффициентов, полученная из навигационных датчиков,
- $x$ — вектор неизвестных параметров траектории,
- $b$ — вектор измеренных сигналов.

Для вычисления решения используется **итерационный метод Якоби**.


### Метод Якоби

На каждой итерации новое значение каждой компоненты вычисляется **только из значений предыдущей итерации**.

Формула обновления имеет вид

$$
x_i^{(k+1)} =
\frac{1}{a_{ii}}
\left(
b_i -
\sum_{j \ne i} a_{ij} x_j^{(k)}
\right),
$$

где

- $a_{ii}$ — диагональный элемент матрицы $A$,
- $a_{ij}$ — остальные элементы строки,
- $x^{(k)}$ — решение на предыдущей итерации.

### Задание

Реализуйте функцию

```python
jacobi(A, b, n)
````

которая:

* принимает

  * `A` — матрицу коэффициентов системы,
  * `b` — вектор правой части,
  * `n` — число итераций метода;

* и возвращает **приближённое решение** $x$ после `n` итераций.

Перед началом итераций необходимо:

* инициализировать вектор решения **нулевым вектором**.

Каждое промежуточное значение решения необходимо **округлять до четырёх знаков после запятой**.

### Интерпретация

Метод Якоби последовательно уточняет параметры навигационной модели, используя значения предыдущей итерации. После нескольких шагов получается приближённое решение системы линейных уравнений.


In [ ]:
import numpy as np

def jacobi(A, b, n):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    x = np.zeros_like(b, dtype=float)

    m = len(b)
    for _ in range(n):
        y = x
        for i in range(m):
            sum1 = np.dot(A[i, :i], y[:i])
            sum2 = np.dot(A[i, i + 1:], y[i + 1:])
            x[i] = round((b[i] - sum1 - sum2) / A[i, i], 4)

    return x

## Задача Б3. Ортонормированный базис для навигационных векторов (3 балла)

Навигационные датчики корабля **X-wing** фиксируют направления на различные космические объекты.  
Каждое измерение можно представить **двумерным вектором** в пространстве координат сенсоров.

Однако для корректной работы навигационного компьютера необходимо построить **ортонормированный базис** пространства, натянутого на эти векторы. Это позволяет избавиться от линейной зависимости измерений и получить независимые направления.

Для этого используется **процесс Грама–Шмидта**.


### Ортогонализация Грама–Шмидта

Пусть задан набор векторов

$$
v_1, v_2, \dots, v_k.
$$

Алгоритм строит ортогональные векторы следующим образом.

Первый вектор нормализуется:

$$
u_1 = \frac{v_1}{\|v_1\|}.
$$

Для каждого следующего вектора вычисляется ортогональная компонента:

$$
w_i =
v_i -
\sum_{j=1}^{i-1}
\langle v_i, u_j \rangle u_j
$$

После этого вектор нормализуется:

$$
u_i = \frac{w_i}{\|w_i\|}.
$$

Если норма полученного вектора слишком мала, то считается, что вектор **линейно зависим**, и он не включается в базис.

### Задание

Реализуйте функцию

```python
orthonormal_basis(vectors, tol)
````

где

* `vectors` — список **двумерных векторов**;
* `tol` — числовой порог, используемый для проверки **линейной независимости**.

Функция должна:

1. применить **процесс Грама–Шмидта**;
2. построить **ортонормированный базис**;
3. вернуть список векторов, которые

   * имеют **единичную длину**,
   * **ортогональны друг другу**,
   * порождают то же подпространство, что и исходные векторы.





In [18]:
import numpy as np

def orthonormal_basis(vectors: list[list[float]], tol: float = 1e-10) -> list[np.ndarray]:
    basis = []

    for vec in vectors:
        v = np.array(vec, dtype=float)

        for q in basis:
            v = v - np.dot(v, q) * q

        norm = np.linalg.norm(v)
        if norm > tol:
            basis.append(v / norm)

    return basis